# Lung Cancer Patient Health and Treatment Records Analysis with PySpark
This notebook demonstrates data cleaning and analysis tasks using PySpark on the provided lung cancer dataset.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('LungCancerAnalysis').getOrCreate()

In [ ]:
df = spark.read.csv('Lung Cancer.csv', header=True, inferSchema=True)
df.show(5)
df.printSchema()
df.show(10, truncate=False)

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  1|64.0|  Male|     Sweden|    2016-04-05|     Stage I|           Yes|Passive Smoker|29.4|              199|           0|     0|        1|           0|  Chemotherapy|        2017-09-10|       0|
|  2|50.0|Female|Netherlands|    2023-04-20|   Stage III|           Yes|Passive Smoker|41.2|              280|           1|     1|        0|           0|       Surgery|        2024-06-17|       1|
|  3|65.0|Femal

## Task 1: Data Cleaning Function

In [ ]:
from pyspark.sql.functions import col, when, lower
from pyspark.sql.types import FloatType, DateType

def clean_data(df):
    df = df.dropDuplicates()
    df = df.withColumn('family_history', when(lower(col('family_history')) == 'yes', 1)
                                         .when(lower(col('family_history')) == 'no', 0)
                                         .otherwise(None))
    num_cols = ['age', 'bmi', 'cholesterol_level', 'hypertension', 'asthma', 'cirrhosis', 'other_cancer', 'survived']
    for c in num_cols:
        df = df.withColumn(c, col(c).cast(FloatType()))
    date_cols = ['diagnosis_date', 'end_treatment_date']
    for c in date_cols:
        df = df.withColumn(c, col(c).cast(DateType()))
    return df

df_clean = clean_data(df)
df_clean.show(5)

+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| 72|54.0|Female|Czech Republic|    2017-06-30|    Stage IV|             0|Passive Smoker|24.0|            160.0|         1.0|   0.0|      0.0|         1.0|       Surgery|        2019-04-06|     0.0|
|190|34.0|  Male|       Estonia|    2015-02-13|     Stage I|             1| Former Smoker|38.8|            252.0|         1.0|   1.0|      1.0|         0.0|      Combined|        2016-12-18|     0.0|


## Task 2: Treatment Duration and Average per Treatment Type

In [7]:
from pyspark.sql.functions import datediff, avg
def avg_treatment_duration(df):
    df = df.withColumn('treatment_duration_days', datediff(col('end_treatment_date'), col('diagnosis_date')))
    return df.groupBy('treatment_type').agg(avg('treatment_duration_days').alias('avg_duration'))
avg_duration_df = avg_treatment_duration(df_clean)
avg_duration_df.show()

+--------------+------------------+
|treatment_type|      avg_duration|
+--------------+------------------+
|     Radiation|458.40320462900917|
|  Chemotherapy|458.39540091909953|
|      Combined| 457.8152186120058|
|       Surgery|457.73744630723684|
+--------------+------------------+



## Task 3: Smoking Status Group with Highest Survival Rate

In [16]:
def highest_survival_smoking_status(df):
    filtered = df.filter(col('survived').isNotNull())
    survival_df = filtered.groupBy('smoking_status').agg(avg(col('survived')).alias('survival_rate'))
    survival_df.show()
    top = survival_df.orderBy(col('survival_rate').desc()).first()
    if top:
        print(f'Highest survival rate group: {top[0]} ({top[1]:.2f})')
    else:
        print('No valid survival data found.')

highest_survival_smoking_status(df_clean)

+--------------+-------------------+
|smoking_status|      survival_rate|
+--------------+-------------------+
|  Never Smoked|0.22091034383684025|
| Former Smoker|0.21964074335789288|
|Current Smoker| 0.2203399760250205|
|Passive Smoker| 0.2200250929784469|
+--------------+-------------------+

Highest survival rate group: Never Smoked (0.22)
Highest survival rate group: Never Smoked (0.22)


## Task 4: Top Three Countries with Highest Percentage of Stage IV Diagnosis

In [9]:
from pyspark.sql.functions import count
def top_countries_stage_iv(df):
    total = df.groupBy('country').agg(count('*').alias('total'))
    stage_iv = df.filter(col('cancer_stage') == 'Stage IV').groupBy('country').agg(count('*').alias('stage_iv_count'))
    joined = total.join(stage_iv, 'country')
    result = joined.withColumn('percentage', col('stage_iv_count') / col('total') * 100)
    return result.orderBy(col('percentage').desc()).select('country', 'percentage').show(3)
top_countries_stage_iv(df_clean)

+--------------+------------------+
|       country|        percentage|
+--------------+------------------+
|        Greece| 25.50223889628464|
|       Croatia|25.427002233085883|
|Czech Republic|25.291166185190818|
+--------------+------------------+
only showing top 3 rows


## Task 5: Filtered Patient Group Analysis

In [15]:
def filtered_patient_stats(df):
    filtered = df.filter(
        (lower(col('gender')) == 'male') &
        (col('cancer_stage').isin('Stage III', 'Stage IV')) &
        (col('family_history') == 1) &
        (lower(col('smoking_status')) == 'current smoker') &
        (col('bmi') > 30) &
        (col('survived') == 1)
    )
    avg_age = filtered.agg(avg('age')).first()[0]
    hypertension_avg = filtered.agg(avg(col('hypertension'))).first()[0]
    if avg_age is not None:
        print(f'Average Age: {avg_age}')
    else:
        print('No matching patients for average age.')
    if hypertension_avg is not None:
        print(f'Percentage with Hypertension: {hypertension_avg * 100:.2f}%')
    else:
        print('No matching patients for hypertension percentage.')

filtered_patient_stats(df_clean)

Average Age: 55.179398872886665
Percentage with Hypertension: 74.77%
